In [12]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.model_selection import cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score

In [3]:
df = pd.read_csv('train.csv',usecols=['Age','Fare','Survived'])
df

,Survived,Age,Fare
0,0,22.0,7.2500
1,1,38.0,71.2833
2,1,26.0,7.9250
3,1,35.0,53.1000
4,0,35.0,8.0500
...,...,...,...
886,0,27.0,13.0000
887,1,19.0,30.0000
888,0,NaN,23.4500
889,1,26.0,30.0000


In [4]:
df.dropna(inplace=True)

In [5]:
df.shape

(714, 3)

In [6]:
df.head()

,Survived,Age,Fare
0,0,22.0,7.2500
1,1,38.0,71.2833
2,1,26.0,7.9250
3,1,35.0,53.1000
4,0,35.0,8.0500


In [7]:
X = df.iloc[:,1:]
y = df.iloc[:,0]

In [8]:
X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=42,test_size=0.2)

In [10]:
X_train.head()

,Age,Fare
328,31.0,20.5250
73,26.0,14.4542
253,30.0,16.1000
719,33.0,7.7750
666,25.0,13.0000


In [11]:
clf = DecisionTreeClassifier()

clf.fit(X_train,y_train)
y_pred = clf.predict(X_test)

In [13]:
accuracy_score(y_test,y_pred)

0.6293706293706294

In [15]:
np.mean(cross_val_score(DecisionTreeClassifier(),X,y,cv=10,scoring='accuracy'))

np.float64(0.6331377151799688)

In [16]:
kbin_age = KBinsDiscretizer(n_bins=15,encode='ordinal',strategy='quantile')
kbin_fare = KBinsDiscretizer(n_bins=15,encode='ordinal',strategy='quantile')

In [17]:
trf = ColumnTransformer([
    ('first',kbin_age,[0]),
    ('second',kbin_fare,[1])
])

In [27]:
X_train_trf = trf.fit_transform(X_train)
X_test_trf = trf.fit_transform(X_test)

In [28]:
trf.named_transformers_['first'].bin_edges_

array([array([ 1.,  9., 16., 17., 20., 21., 24., 26., 28., 30., 34., 36., 39.,
              44., 52., 62.])                                                 ],
      dtype=object)

In [29]:
trf.named_transformers_['second'].bin_edges_

array([array([  0.    ,   7.125 ,   7.775 ,   7.925 ,   8.05  ,   9.5   ,
               10.5   ,  13.    ,  17.8   ,  22.525 ,  26.3875,  31.275 ,
               39.    ,  53.1   ,  79.65  , 512.3292])                   ],
      dtype=object)

In [30]:
output = pd.DataFrame({
    'age':X_train['Age'],
    'age_trf':X_train_trf[:,0],
    'fare':X_train['Fare'],
    'fare_trf':X_train_trf[:,1]
})

In [31]:
output.head()

,age,age_trf,fare,fare_trf
328,31.0,8.0,20.5250,8.0
73,26.0,6.0,14.4542,7.0
253,30.0,8.0,16.1000,7.0
719,33.0,9.0,7.7750,2.0
666,25.0,6.0,13.0000,6.0


In [33]:
output['age_laabels'] = pd.cut(
    x = X_train['Age'],
    bins=trf.named_transformers_['first'].bin_edges_[0].tolist()
)

output['fare_laabels'] = pd.cut(
    x = X_train['Fare'],
    bins=trf.named_transformers_['second'].bin_edges_[0].tolist()
)

In [34]:
output.head()

,age,age_trf,fare,fare_trf,age_laabels,fare_laabels
328,31.0,8.0,20.5250,8.0,"(30.0, 34.0]","(17.8, 22.525]"
73,26.0,6.0,14.4542,7.0,"(24.0, 26.0]","(13.0, 17.8]"
253,30.0,8.0,16.1000,7.0,"(28.0, 30.0]","(13.0, 17.8]"
719,33.0,9.0,7.7750,2.0,"(30.0, 34.0]","(7.125, 7.775]"
666,25.0,6.0,13.0000,6.0,"(24.0, 26.0]","(10.5, 13.0]"


In [35]:
clf = DecisionTreeClassifier()
clf.fit(X_train_trf,y_train)
y_pred2 = clf.predict(X_test_trf)

In [36]:
accuracy_score(y_test,y_pred2)

0.6573426573426573

In [37]:
X_trf = trf.fit_transform(X)
np.mean(cross_val_score(DecisionTreeClassifier(),X,y,cv=10,scoring='accuracy'))

np.float64(0.6373239436619718)